In [1]:
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)

In [2]:
MODEL_PATH = "Model/surya-kavach.joblib"
start_time = time.time()
model = joblib.load(MODEL_PATH)
loading_time =  time.time() - start_time
print(f"\nModel loaded in {round(loading_time, 5)} seconds.") 


Model loaded in 0.08807 seconds.


In [3]:
df = pd.read_parquet("../Dataset/final-dataset/dataset-with-features.parquet")
model_df = df.dropna(subset=['target_log_flux_1h']).copy()

test_df = model_df.loc["2019"]

TARGET = 'target_log_flux_1h'
ignore_col = [TARGET, 'E2W_COR_FLUX']
feature_col = [col for col in test_df.columns if col not in ignore_col]

X_test = test_df[feature_col]
y_test_log = test_df[TARGET]
y_test_raw = (10 ** y_test_log) - 1 

In [4]:
start_time = time.time()
y_pred_log = model.predict(X_test)
inference_time = time.time() - start_time 
y_pred_raw = np.maximum(0, ((10 ** y_pred_log) - 1))
print(f"\nGenerated predictions for {len(X_test):,} test hours in {inference_time:.2f} seconds.")


Generated predictions for 364,469 test hours in 0.58 seconds.


In [5]:
print("\nTimeSeries Forecasting Model Evalution Metrics")

log_r2 = r2_score(y_test_log, y_pred_log) * 100
log_mae = mean_absolute_error(y_test_log, y_pred_log)
log_rmse = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
print("\nLog scale evaluation")
print(f"R2: {round(log_r2, 5)}, MAE: {round(log_mae, 5)}, RMSE: {round(log_rmse, 5)}")

raw_r2 = r2_score(y_test_raw, y_pred_raw) * 100
raw_mae = mean_absolute_error(y_test_raw, y_pred_raw)
raw_rmse = np.sqrt(mean_squared_error(y_test_raw, y_pred_raw))
print("\nRaw scale evaluation")
print(f"R2: {round(raw_r2, 5)}, MAE: {round(raw_mae, 5)}, RMSE: {round(raw_rmse, 5)}")
    


TimeSeries Forecasting Model Evalution Metrics

Log scale evaluation
R2: 96.60218, MAE: 0.09588, RMSE: 0.14438

Raw scale evaluation
R2: 91.77592, MAE: 750.67737, RMSE: 3657.25676


In [6]:
# NOAA strom threshlod values
noaa_thresholds = {
    "S1 (Minor)": 10.0,
    "S2 (Moderate)": 100.0,
    "S3 (Strong)": 1000.0,
    "S4 (Severe)": 10000.0,
}

In [7]:
metrics = []
for name, thresh in noaa_thresholds.items():
    yt, yp = (y_test_raw >= thresh), (y_pred_raw >= thresh)
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

    rec = recall_score(yt, yp, zero_division=0) * 100
    prec = precision_score(yt, yp, zero_division=0) * 100
    f1 = f1_score(yt, yp, zero_division=0)

    metrics.append({
        "Level": name,
        "Threshold": f">={thresh:,.0f} pfu",
        "Actual Hours": f"{tp + fn:,}",
        "TP (Caught)": f"{tp:,}",
        "FN (Missed)": f"{fn:,}",
        "FP (False Alarm)": f"{fp:,}",
        "Recall": f"{rec:.1f}%",
        "Precision": f"{prec:.1f}%",
        "F1-Score": f"{f1:.4f}",
    })

df_metrics = pd.DataFrame(metrics)
print(df_metrics.to_string(index=False))

        Level    Threshold Actual Hours TP (Caught) FN (Missed) FP (False Alarm) Recall Precision F1-Score
   S1 (Minor)     >=10 pfu      364,465     364,465           0                4 100.0%    100.0%   1.0000
S2 (Moderate)    >=100 pfu      276,640     267,253       9,387           10,089  96.6%     96.4%   0.9648
  S3 (Strong)  >=1,000 pfu      112,369     105,624       6,745            6,342  94.0%     94.3%   0.9417
  S4 (Severe) >=10,000 pfu       25,854      23,281       2,573            2,556  90.0%     90.1%   0.9008
